In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import abs as spark_abs
from pyspark.sql.functions import (
    col, trim, current_timestamp, when
)


def prepare_sales():

    return (
        spark.readStream.table("sales_bronze")
        .withColumn("sale_id", trim(col("sale_id")))
        .withColumn("customer_id", trim(col("customer_id")))
        .withColumn("product_name", trim(col("product_name")))
        .withColumn("city", trim(col("city")))
        .withColumn("sale_timestamp", col("sale_timestamp").cast("timestamp"))
        .withColumn("quantity", col("quantity").cast("int"))
        .withColumn("unit_price", col("unit_price").cast("double"))
        .withColumn("total_amount", col("total_amount").cast("double"))
        .withColumn("_processed_at", current_timestamp())
    )


def invalid_sales_condition():

    return (
        col("sale_id").isNull()
        | (col("sale_id") == "")
        | col("customer_id").isNull()
        | (col("customer_id") == "")
        | col("product_name").isNull()
        | (col("product_name") == "")
        | col("city").isNull()
        | (col("city") == "")
        | col("sale_timestamp").isNull()
        | col("quantity").isNull()
        | (col("quantity") <= 0)
        | col("unit_price").isNull()
        | (col("unit_price") <= 0)
        | col("total_amount").isNull()
        | (col("total_amount") <= 0)
        | (spark_abs(col("total_amount") - (col("quantity") * col("unit_price")) ) >= 0.01)


@dp.table(
    name="sales_silver",
    comment="Cleaned and validated sales records"
)
@dp.expect_or_drop("valid_sale_id", "sale_id IS NOT NULL AND sale_id <> ''")
@dp.expect_or_drop("valid_customer_id", "customer_id IS NOT NULL AND customer_id <> ''")
@dp.expect_or_drop("valid_quantity", "quantity > 0")
@dp.expect_or_drop("valid_unit_price", "unit_price > 0")
@dp.expect_or_drop("valid_total_amount", "total_amount > 0")
@dp.expect_or_drop("valid_sale_timestamp", "sale_timestamp IS NOT NULL")
@dp.expect_or_drop(
    "valid_total_calculation",
    "ABS(total_amount - (quantity * unit_price)) < 0.01"
)
@dp.expect_or_drop(
    "valid_product",
    "product_name IS NOT NULL AND product_name <> ''"
)
@dp.expect_or_drop(
    "valid_city",
    "city IS NOT NULL AND city <> ''"
)
def sales_silver():

    return prepare_sales().filter(~invalid_sales_condition())


@dp.table(
    name="sales_quarantine",
    comment="Invalid sales records separated for investigation"
)
def sales_quarantine():

    df = prepare_sales()

    return (
        df
        .filter(invalid_sales_condition())
        .withColumn(
            "_quarantine_reason",
            when(col("sale_id").isNull(), "Missing sale_id")
            .when(col("customer_id").isNull(), "Missing customer_id")
            .when(col("quantity").isNull() | (col("quantity") <= 0), "Invalid quantity")
            .when(col("unit_price").isNull() | (col("unit_price") <= 0), "Invalid unit_price")
            .when(col("total_amount").isNull() | (col("total_amount") <= 0), "Invalid total_amount")
            .when(spark_abs(col("total_amount") - (col("quantity") * col("unit_price")) ) >= 0.01, "Total amount calculation mismatch")
            .otherwise("Other validation failure")
        )
    )